# 04_02 – Доступность признаков и утечка исхода

Рассматриваем прогноз до первой наблюдаемой попытки. Согласованный момент прогноза – вход C40; его сопоставление с первой попыткой требует проверки. [Описание авторов набора](https://zenodo.org/records/10210178) проверено 08.10.2026. Названия и физические типы берем из последней рабочей копии; интерпретации доступности – наши исследовательские решения с явно указанными ограничениями.

Полнота поля не доказывает доступность до события. Целевой аэропорт нельзя автоматически заменить аэропортом конечной посадки. Справочник, проверенный в 2026 году, не становится историческим справочником 2019–2023.

In [1]:
# Находим проект и подключаем диагностику без зависимости от предыдущего ядра
from pathlib import Path
import runpy
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
module = runpy.run_path(str(ROOT / 'europe/scripts/audit_feature_availability.py'))

# Проверяем все строки и сохраняем карту полей и счетчики без фильтрации базы
feature_map, c40_coverage = module['main']()
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 100):
    display(feature_map)

{
  "input": "europe\\data\\working\\03_c40_diagnostic_flags.parquet",
  "input_sha256": "ddf11438bab44137242ffcf4da928418e116efd46defce58dc8a388b67942b6d",
  "input_unchanged": true,
  "rows_checked": 7984254,
  "go_arounds": 26253,
  "physical_columns": 31,
  "rows_changed": 0,
  "rows_deleted": 0,
  "prediction_context": "before first observed attempt",
  "feature_set_finalized": false,
  "c40_filter_applied": false,
  "timing_condition": "C40 <= first attempt; equality retained, no lead time guaranteed",
  "role_counts": {
    "provenance": 6,
    "identifier": 5,
    "outcome": 4,
    "metadata_candidate": 4,
    "event_candidate": 4,
    "context_candidate": 2,
    "outcome_diagnostic": 2,
    "target": 1,
    "unresolved": 1,
    "reference_candidate": 1,
    "diagnostic": 1
  },
  "source_url": "https://zenodo.org/records/10210178",
  "source_checked_date": "2026-10-08"
}


,field,physical_type,null_rows,null_percent,exact_empty_string_rows,role,availability,reason,required_verification
0,flight_id,string,0,0.000000,0,identifier,не включать в базовый набор,"Идентификация, возможное запоминание бортов и рейсов",Использовать для связей и проверки пересечения групп; flight_id не уникален
1,callsign,string,0,0.000000,0,identifier,не включать в базовый набор,"Идентификация, возможное запоминание бортов и рейсов",Использовать для связей и проверки пересечения групп; flight_id не уникален
2,icao24,string,0,0.000000,0,identifier,не включать в базовый набор,"Идентификация, возможное запоминание бортов и рейсов",Использовать для связей и проверки пересечения групп; flight_id не уникален
3,landing_time,"timestamp[ns, tz=UTC]",0,0.000000,0,outcome,исключить из входов,Поле завершенного наблюдения содержит информацию об исходе,"Можно использовать для аудита и определения метки, не для раннего прогноза"
4,registration,string,1101304,13.793449,0,identifier,не включать в базовый набор,"Идентификация, возможное запоминание бортов и рейсов",Использовать для связей и проверки пересечения групп; flight_id не уникален
5,typecode,string,1334092,16.709038,0,metadata_candidate,условно доступно,Справочное обогащение без гарантии исторической доступности,"Источник, версия, дата действия и возможность получить поле до исхода"
6,icaoaircrafttype,string,1402236,17.562517,0,metadata_candidate,условно доступно,Справочное обогащение без гарантии исторической доступности,"Источник, версия, дата действия и возможность получить поле до исхода"
7,airport,string,0,0.000000,0,context_candidate,условно доступно,В таблице указан аэропорт конечной посадки и его страна,"Подтвердить, что это целевой аэропорт в момент прогноза, а не результат диверсии"
8,country,string,0,0.000000,0,context_candidate,условно доступно,В таблице указан аэропорт конечной посадки и его страна,"Подтвердить, что это целевой аэропорт в момент прогноза, а не результат диверсии"
9,attempt_times,"list<element: timestamp[us, tz=UTC]>",0,0.000000,0,outcome,исключить из входов,Поле завершенного наблюдения содержит информацию об исходе,"Можно использовать для аудита и определения метки, не для раннего прогноза"


## Роли полей

Разделяем целевую метку, сведения об исходе, идентификаторы, следы подготовки и условные кандидаты. Для каждого кандидата указана недостающая проверка. Категория «условно доступно» не означает разрешение включить поле в модель. Флаги C40 после первой попытки/посадки используют будущие времена и исключаются из раннего прогноза.

In [2]:
# Сводим роли и отдельно показываем поля с информацией об исходе
display(feature_map.groupby(['role', 'availability']).size().rename('fields').reset_index())
with pd.option_context('display.max_colwidth', 100):
    display(feature_map.loc[feature_map['role'].isin(['outcome', 'outcome_diagnostic']),
                           ['field', 'reason']])

,role,availability,fields
0,context_candidate,условно доступно,2
1,diagnostic,не включать в базовый набор,1
2,event_candidate,условно доступно в точке C40,4
3,identifier,не включать в базовый набор,5
4,metadata_candidate,условно доступно,4
5,outcome,исключить из входов,4
6,outcome_diagnostic,исключить из входов,2
7,provenance,не включать в базовый набор,6
8,reference_candidate,не допускать до проверки,1
9,target,не вход модели,1


,field,reason
3,landing_time,Поле завершенного наблюдения содержит информацию об исходе
9,attempt_times,Поле завершенного наблюдения содержит информацию об исходе
10,ILS,Поле завершенного наблюдения содержит информацию об исходе
11,n_attempts,Поле завершенного наблюдения содержит информацию об исходе
29,c40_after_first_attempt,Флаг рассчитан относительно времен завершенного наблюдения
30,c40_after_landing,Флаг рассчитан относительно времен завершенного наблюдения


## Что изменит требование C40

Считаем последствия нескольких диагностических условий на всей базе. Группы пересекаются и не должны суммироваться. «Не позже первой попытки» включает равенство времен и не гарантирует запас времени для прогноза. Полнота четырех полей и конечность чисел не доказывают правильность единиц, координат или присоединения.

Эти условия используют ретроспективную информацию для аудита. Их нельзя подавать в модель как признаки; исключение строк по ним может изменить состав целевого класса.

In [3]:
# Сравниваем покрытие всей базы и отдельно покрытие положительного класса
with pd.option_context('display.max_columns', None):
    display(c40_coverage)

# Показываем условных кандидатов вместе с пропусками и необходимыми проверками
candidate_roles = ['context_candidate', 'metadata_candidate', 'event_candidate', 'reference_candidate']
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 100):
    display(feature_map.loc[feature_map['role'].isin(candidate_roles),
                           ['field', 'null_percent', 'availability', 'required_verification']])

,diagnostic_group,rows,go_arounds,go_around_percent,row_coverage_percent,positive_coverage_percent
0,all_rows,7984254,26253,0.328810,100.000000,100.000000
1,c40_time_missing,1652382,7847,0.474890,20.695509,29.889917
2,c40_time_present,6331872,18406,0.290688,79.304491,70.110083
3,c40_time_present_not_after_first_attempt,6331785,18361,0.289981,79.303401,69.938674
4,c40_all_four_fields_present,6331872,18406,0.290688,79.304491,70.110083
5,c40_complete_finite_not_after_first_attempt,6331785,18361,0.289981,79.303401,69.938674
6,c40_after_first_attempt,87,45,51.724138,0.001090,0.171409
7,c40_after_landing,42,0,0.000000,0.000526,0.000000


,field,null_percent,availability,required_verification
5,typecode,16.709038,условно доступно,"Источник, версия, дата действия и возможность получить поле до исхода"
6,icaoaircrafttype,17.562517,условно доступно,"Источник, версия, дата действия и возможность получить поле до исхода"
7,airport,0.000000,условно доступно,"Подтвердить, что это целевой аэропорт в момент прогноза, а не результат диверсии"
8,country,0.000000,условно доступно,"Подтвердить, что это целевой аэропорт в момент прогноза, а не результат диверсии"
13,market_segment,18.978304,условно доступно,"Источник, версия, дата действия и возможность получить поле до исхода"
14,AC_CLASS,20.695509,условно доступно,"Источник, версия, дата действия и возможность получить поле до исхода"
16,C40_CROSS_TIME,20.695509,условно доступно в точке C40,"Порядок времен, полнота, единицы, сопоставление и доступность в реальном времени"
17,C40_BEARING,20.695509,условно доступно в точке C40,"Порядок времен, полнота, единицы, сопоставление и доступность в реальном времени"
18,C40_CROSS_LAT,20.695509,условно доступно в точке C40,"Порядок времен, полнота, единицы, сопоставление и доступность в реальном времени"
19,C40_CROSS_LON,20.695509,условно доступно в точке C40,"Порядок времен, полнота, единицы, сопоставление и доступность в реальном времени"


## Решение шага 04_02

Карта охватывает все физические поля. Данные не изменены, фильтрация C40 не применялась и окончательный набор признаков не утвержден. Погодных полей и полных траекторий в текущем файле нет; их наличие нельзя предполагать.

Далее 04_03 – оценка исследовательской популяции и покрытия по аэропортам/времени. Согласовано: все наблюдаемые полеты, включая специальные; прогноз в точке входа C40. Проверяем сопоставление и покрытие по группам. Только после этого оцениваем конкретные правила отбора и создаем аналитическую копию.

[Карта полей и счетчики](../../reports/analytical_sample/04_02_review.md), [постановка задачи](04_01_task_definition.ipynb).